In [41]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta
import numpy as np

In [42]:
conn = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=master;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={conn}")

query = """
SELECT 
    Cliente,
    CAST(Fecha AS DATE) AS Fecha_Compra
FROM Ventas_Comerssia.dbo.Ventas_Unificadas
"""
df = pd.read_sql(query, engine)
df["Fecha_Compra"] = pd.to_datetime(df["Fecha_Compra"])

In [43]:
# ===============================
# 3. Funciones auxiliares
# ===============================
def calcular_metricas(df, inicio_mes, fin_mes):
    """
    df debe tener columnas: Cliente, Fecha_Compra (datetime)
    inicio_mes = primer día del mes
    fin_mes    = último día del mes
    """

    # Cortes
    corte_inicio = inicio_mes - pd.Timedelta(days=1)  # último día del mes anterior
    corte_fin = fin_mes

    # Ventanas de 365 días
    inicio_ventana = corte_inicio - pd.Timedelta(days=365)
    inicio_ventana_fin = corte_inicio

    fin_ventana = corte_fin - pd.Timedelta(days=365)
    fin_ventana_fin = corte_fin

    # ================================
    # Activos inicio
    # ================================
    ultima_inicio = (
        df[df["Fecha_Compra"] <= corte_inicio]
        .groupby("Cliente")["Fecha_Compra"].max()
    )
    activos_inicio = ultima_inicio[
        (ultima_inicio >= inicio_ventana) & (ultima_inicio <= inicio_ventana_fin)
    ].index

    # ================================
    # Activos fin
    # ================================
    ultima_fin = (
        df[df["Fecha_Compra"] <= corte_fin]
        .groupby("Cliente")["Fecha_Compra"].max()
    )
    activos_fin = ultima_fin[
        (ultima_fin >= fin_ventana) & (ultima_fin <= fin_ventana_fin)
    ].index

    # ================================
    # Nuevos
    # ================================
    primera_compra = df.groupby("Cliente")["Fecha_Compra"].min()
    nuevos = primera_compra[
        (primera_compra >= inicio_mes) & (primera_compra <= fin_mes)
    ].index

    # ================================
    # Churn
    # ================================
    churn = set(activos_inicio) - set(activos_fin)

    # ================================
    # Reactivados
    # ================================
    compradores_mes = df[
        (df["Fecha_Compra"] >= inicio_mes) & (df["Fecha_Compra"] <= fin_mes)
    ]["Cliente"].unique()

    reactivados = []
    for cliente in compradores_mes:
        ultima_antes = df[
            (df["Cliente"] == cliente) & (df["Fecha_Compra"] < inicio_mes)
        ]["Fecha_Compra"].max()
        if pd.notnull(ultima_antes):
            if (inicio_mes - ultima_antes).days > 365:
                reactivados.append(cliente)

    # ================================
    # Construir resultados
    # ================================
    return {
        "Periodo": inicio_mes.strftime("%Y-%m"),
        "StockClientes": df["Cliente"].nunique(),
        "ClientesActivosInicio": len(activos_inicio),
        "ClientesNuevos": len(nuevos),
        "ClientesChurn": len(churn),
        "ClientesReactivados": len(np.unique(reactivados)),
        "ClientesActivosFin": len(activos_fin),
        "ChurnRate": round(len(churn) / len(activos_inicio), 4) if len(activos_inicio) > 0 else 0
    }



In [44]:
# ==================================
# 2. Ejecutar para un rango de meses
# ==================================
def generar_tabla(df, inicio_periodo, fin_periodo):
    """
    inicio_periodo, fin_periodo en formato 'YYYY-MM'
    """
    periodos = pd.date_range(f"{inicio_periodo}-01", f"{fin_periodo}-01", freq="MS")
    resultados = []

    for inicio in periodos:
        fin = inicio + pd.offsets.MonthEnd(0)
        resultados.append(calcular_metricas(df, inicio, fin))

    return pd.DataFrame(resultados)

In [45]:
# Generar tabla enero–agosto 2025
tabla = generar_tabla(df, "2025-01", "2025-08")
print(tabla)

   Periodo  StockClientes  ClientesActivosInicio  ClientesNuevos  \
0  2025-01         271597                  85522            2188   
1  2025-02         271597                  84989            1651   
2  2025-03         271597                  84926            2124   
3  2025-04         271597                  84702            1902   
4  2025-05         271597                  84308            2823   
5  2025-06         271597                  83944            2110   
6  2025-07         271597                  84156            2080   
7  2025-08         271597                  84322            2189   

   ClientesChurn  ClientesReactivados  ClientesActivosFin  ChurnRate  
0           4367                 1646               84989     0.0511  
1           2839                 1128               84926     0.0334  
2           4046                 1703               84702     0.0476  
3           3887                 1596               84308     0.0459  
4           5312                

In [46]:
# ===============================
# 5. Exportar a Excel (opcional)
# ===============================
tabla.to_excel("resumen_clientes.xlsx", index=False)
print(tabla)

   Periodo  StockClientes  ClientesActivosInicio  ClientesNuevos  \
0  2025-01         271597                  85522            2188   
1  2025-02         271597                  84989            1651   
2  2025-03         271597                  84926            2124   
3  2025-04         271597                  84702            1902   
4  2025-05         271597                  84308            2823   
5  2025-06         271597                  83944            2110   
6  2025-07         271597                  84156            2080   
7  2025-08         271597                  84322            2189   

   ClientesChurn  ClientesReactivados  ClientesActivosFin  ChurnRate  
0           4367                 1646               84989     0.0511  
1           2839                 1128               84926     0.0334  
2           4046                 1703               84702     0.0476  
3           3887                 1596               84308     0.0459  
4           5312                